# ARCHS4 model building with CLAMP

💡 **Environment:** `clamp-analyses`  

This notebook performs Singular Value Decomposition (SVD) on the ARCHS4 gene expression dataset. The goal is to reduce dimensionality and extract principal components that capture the most variance in the data.

## Load libraries

In [ ]:
if (!requireNamespace("PLIER", quietly = TRUE)) {
    devtools::install_github("wgmao/PLIER")
}

if (!requireNamespace("CLAMP", quietly = TRUE)) {
    REPO_PATH <- "/home/msubirana/Documents/pivlab/CLAMP" 
    remotes::install_local(REPO_PATH, force = TRUE, dependencies = FALSE)
}

library(here)

source(here("config.R"))

set.seed(config$ARCHS4$RANDOM_SVD_SEED)

here() starts at /home/msubirana/Documents/pivlab/clamp-analyses



## Output directory

In [2]:
output_dir <- config$ARCHS4$DATASET_FOLDER
dir.create(output_dir, showWarnings = FALSE, recursive = TRUE)

## Input data

In [ ]:
meta <- readRDS(file.path(output_dir, "metadata_filtered.rds"))
n_genes_thin <- meta$n_genes_thin
n_samples <- meta$n_samples

In [ ]:
fbm_file  <- file.path(output_dir, "fbm")
output_file <- paste0(fbm_file, "_filtered")

fbm_obj_filtered <- FBM(
  nrow        = n_genes_thin,
  ncol        = n_samples,
  backingfile = output_file ,
  create_bk   = FALSE,
)

## SVD computation and SVD K estimation

In [ ]:
message(SVD_K)
message("Using SVD K = ", SVD_K)

Using SVD K = 699



In [ ]:
output_file <- file.path(output_dir, "svd_full.rds")

N_CORES <- config$ARCHS4$PLIER_PARAMS$RANDOM_SVD_N_CORES
if (N_CORES > 1) {
  # if we are parallelizing, then disable BLAS parallelization
  options(bigstatsr.check.parallel.blas = FALSE)
  blas_nproc <- getOption("default.nproc.blas")
  options(default.nproc.blas = NULL)
}

fbm_obj.svd=big_randomSVD(
  fbm_obj_filtered,
  k = SVD_K,
  ncores = N_CORES
)

if (N_CORES > 1) {
  # restore previous state
  options(bigstatsr.check.parallel.blas = TRUE)
  options(default.nproc.blas = blas_nproc)
}

saveRDS(fbm_obj.svd, file = output_file)

In [ ]:
# remove NaN values (if present)
output_file <- file.path(output_dir, "svd.rds")

valid_idx <- which(!is.nan(fbm_obj.svd$d))
fbm_obj.svd$d <- fbm_obj.svd$d[valid_idx]
fbm_obj.svd$u <- fbm_obj.svd$u[, valid_idx, drop = FALSE]
fbm_obj.svd$v <- fbm_obj.svd$v[, valid_idx, drop = FALSE]

saveRDS(fbm_obj.svd, file = output_file)

In [ ]:
svd_list <- list(d=fbm_obj.svd$d)
CLAMP_K <- num.pc(svd_list) * 2

message(paste0("K inferred for PLIER: ", CLAMP_K))